In [ ]:
from pathlib import Path
import math
import pandas as pd

DATA = Path(r"C:\Users\User\Desktop\Sanuth\data")
PROC = DATA / "processed"

rents = pd.read_csv(PROC / "rents_by_suburb.csv", dtype={"SAL_CODE21": str})
tt = pd.read_csv(PROC / "travel_times.csv", dtype={"from_id": str})
points = pd.read_csv(PROC / "suburb_points.csv", dtype={"id": str})
campuses = pd.read_csv(PROC / "campuses.csv")
print(len(rents), "rent rows |", len(tt), "travel time rows |", len(points), "suburbs |", len(campuses), "campuses")

In [ ]:
OPTIONS = {
    "1 bedroom flat": ("Own 1-bedroom flat", 1),
    "2 bedroom flat": ("Room in a 2-bedroom flat share", 2),
    "3 bedroom house": ("Room in a 3-bedroom house share", 3),
}
r = rents[rents.property_type.isin(OPTIONS)].copy()
r["option"] = r.property_type.map(lambda p: OPTIONS[p][0])
r["rent_pp"] = (r["median"] / r.property_type.map(lambda p: OPTIONS[p][1])).round(0)
r = r[["SAL_CODE21", "suburb", "suburb_group", "region", "option", "median", "count", "rent_pp"]]
r.groupby("option")["rent_pp"].describe()[["count", "min", "50%", "max"]]

In [ ]:
def km(lat1, lon1, lat2, lon2):
    p = math.pi / 180
    a = (math.sin((lat2 - lat1) * p / 2) ** 2
         + math.cos(lat1 * p) * math.cos(lat2 * p) * math.sin((lon2 - lon1) * p / 2) ** 2)
    return 12742 * math.asin(math.sqrt(a))

t = tt.rename(columns={"from_id": "SAL_CODE21", "to_id": "campus"})[["SAL_CODE21", "campus", "travel_time"]]
t = t.merge(points.rename(columns={"id": "SAL_CODE21"})[["SAL_CODE21", "lat", "lon"]], on="SAL_CODE21")
t = t.merge(campuses.rename(columns={"lat": "c_lat", "lon": "c_lon"}), on="campus")
t["distance_km"] = [round(km(a, b, c, d), 1) for a, b, c, d in zip(t.lat, t.lon, t.c_lat, t.c_lon)]

scores = r.merge(t[["SAL_CODE21", "campus", "travel_time", "distance_km"]], on="SAL_CODE21")
print(len(scores), "rows;", scores.travel_time.isna().sum(), "with no public transport trip under 2 hours")
scores = scores.dropna(subset=["travel_time"])

In [ ]:
DAYS, VALUE_OF_TIME = 4, 15

scores["commute_cost"] = (scores.travel_time / 60 * DAYS * 2 * VALUE_OF_TIME).round(0)
scores["weekly_cost"] = scores.rent_pp + scores.commute_cost
scores["rank"] = scores.groupby(["campus", "option"]).weekly_cost.rank(method="min").astype(int)
scores = scores.sort_values(["campus", "option", "rank"])
scores.to_csv(PROC / "suburb_scores.csv", index=False)
print("saved", len(scores), "rows to processed/suburb_scores.csv")
scores.head(10)

In [ ]:
share = scores[scores.option == "Room in a 3-bedroom house share"]
for campus, g in share.groupby("campus"):
    print(f"\n{campus}")
    print(g.nsmallest(5, "weekly_cost")[["suburb", "rent_pp", "travel_time", "weekly_cost"]].to_string(index=False))

In [ ]:
cheap_line = scores.groupby("option").rent_pp.quantile(0.25).rename("cheap_line")
s = scores.merge(cheap_line, on="option")
gems = s[(s.rent_pp <= s.cheap_line) & (s.travel_time <= 40)]
traps = scores[(scores.distance_km <= 12) & (scores.travel_time >= 50)].drop_duplicates(["campus", "suburb"])

print("HIDDEN GEMS (room in a 3-bedroom share)")
print(gems[gems.option == "Room in a 3-bedroom house share"]
      .sort_values(["campus", "weekly_cost"])[["campus", "suburb", "rent_pp", "travel_time"]].to_string(index=False))
print("\nTRAPS")
print(traps.sort_values(["campus", "travel_time"], ascending=[True, False])
      [["campus", "suburb", "distance_km", "travel_time"]].to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

CAMPUS = "Monash Clayton"          # change to any campus name
OPTION = "Room in a 3-bedroom house share"
g = scores[(scores.campus == CAMPUS) & (scores.option == OPTION)]
gem_names = set(gems[(gems.campus == CAMPUS) & (gems.option == OPTION)].suburb)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(g.travel_time, g.rent_pp, color="#999", alpha=0.6, label="Suburb")
hl = g[g.suburb.isin(gem_names)]
ax.scatter(hl.travel_time, hl.rent_pp, color="#1a9850", s=60, label="Hidden gem")
for _, row in hl.iterrows():
    ax.annotate(row.suburb, (row.travel_time, row.rent_pp), xytext=(4, 4), textcoords="offset points", fontsize=9)
ax.set_xlabel("Public transport time to campus (minutes, Tue 8-9am)")
ax.set_ylabel("Rent per person ($ per week)")
ax.set_title(f"{CAMPUS}: rent vs commute ({OPTION})")
ax.legend()
plt.show()